# Tutorial 05 — Inference

Covers:
1. **`GenerationConfig`** — sampling parameters
2. **`generate_from_embedding`** — generate from a precomputed structure embedding
3. **`write_fasta`** — save sequences to FASTA
4. **Batch generation demo** — generate 10 diverse sequences and inspect them

No checkpoint required — uses a randomly-initialised model.

In [ ]:
import sys, pathlib, tempfile
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import torch
torch.manual_seed(7)

from src.utils.tokenizer import AntibodyTokenizer
from src.models.decoder import AntibodyDecoder
from src.models.pooling import build_pooler
from src.models.projectors import build_projector
from src.models.abllava import AbLLaVA

tok = AntibodyTokenizer()

## 1. Build a tiny model for demo

In [ ]:
d_model = 128
decoder   = AntibodyDecoder(vocab_size=tok.vocab_size, d_model=d_model, n_layers=2, n_heads=4, d_ff=256)
projector = build_projector('mlp', d_in=512, d_out=d_model)
model     = AbLLaVA(decoder=decoder, projector=projector, pooler_name='cdr_mean')
model.eval()
print(f"Model ready  ({sum(p.numel() for p in model.parameters())/1e6:.2f}M params)")

## 2. GenerationConfig

In [ ]:
from src.inference.generate import GenerationConfig, generate_from_embedding

cfg_greedy = GenerationConfig(
    max_new_tokens=120,
    temperature=1.0,
    top_p=1.0,
    top_k=None,
    do_sample=False,
    num_return_sequences=1,
)

cfg_sample = GenerationConfig(
    max_new_tokens=120,
    temperature=0.8,
    top_p=0.9,
    top_k=50,
    do_sample=True,
    num_return_sequences=5,
)

print("Greedy config:", cfg_greedy)
print("Sampling config:", cfg_sample)

## 3. Generate from a structure embedding

In [ ]:
# Synthetic structure embedding (as from AntiFold / CachedEmbeddingEncoder)
N = 230
struct_emb = torch.randn(1, N, 512)       # (1, N, d_e)
cdr_spans  = torch.tensor([[
    [26,32],[50,57],[95,102],[143,153],[169,173],[208,216]
]], dtype=torch.long)                      # (1, 6, 2)
pad_mask   = torch.ones(1, N, dtype=torch.bool)
plddt      = torch.rand(1, N) * 30 + 70

sequences = generate_from_embedding(
    model=model,
    tokenizer=tok,
    struct_emb=struct_emb,
    cdr_spans=cdr_spans,
    pad_mask=pad_mask,
    plddt=plddt,
    gen_cfg=cfg_sample,
    device='cpu',
)

print(f"Generated {len(sequences)} sequence pairs:")
for i, (h, l) in enumerate(sequences):
    print(f"  [{i}] heavy={h[:20]}...({len(h)})  light={l[:20]}...({len(l)})")

## 4. Write to FASTA

In [ ]:
from src.inference.io import write_fasta

with tempfile.TemporaryDirectory() as tmpdir:
    fasta_path = pathlib.Path(tmpdir) / 'generated.fasta'
    write_fasta(sequences, fasta_path)
    content = fasta_path.read_text()
    print(content[:600])
    lines = [l for l in content.strip().split('\n') if l.startswith('>')]
    print(f"... ({len(lines)} sequences written)")

## 5. Diversity across generated sequences

In [ ]:
from src.eval.sequence import levenshtein, compute_diversity

heavies = [h for h, _ in sequences if len(h) > 0]
if len(heavies) >= 2:
    div = compute_diversity(heavies)
    print(f"Heavy-chain diversity (mean normalised Levenshtein): {div:.3f}")
    
    print("\nPairwise Levenshtein distances:")
    for i in range(min(3, len(heavies))):
        for j in range(i+1, min(3, len(heavies))):
            d = levenshtein(heavies[i], heavies[j])
            print(f"  seq{i} vs seq{j}: {d} edits")
else:
    print("Not enough non-empty sequences for diversity (expected with random model)")

## 6. Temperature effect on diversity

In [ ]:
print("Temperature → diversity (higher T = more diverse):")
for temp in [0.5, 0.8, 1.0, 1.2]:
    cfg_t = GenerationConfig(max_new_tokens=80, temperature=temp, top_p=0.9, do_sample=True, num_return_sequences=5)
    seqs_t = generate_from_embedding(
        model=model, tokenizer=tok,
        struct_emb=struct_emb, cdr_spans=cdr_spans,
        pad_mask=pad_mask, plddt=plddt,
        gen_cfg=cfg_t, device='cpu',
    )
    hs = [h for h, _ in seqs_t if len(h) > 1]
    div = compute_diversity(hs) if len(hs) >= 2 else float('nan')
    print(f"  T={temp:.1f}  n_valid={len(hs)}  diversity={div:.3f}")

---
**Summary**: Inference runs as `structure embedding → pool → project → decoder.generate()`. Temperature and top-p control sequence diversity. In production, replace the random embedding with output from `CachedEmbeddingEncoder` or a live `AntiFoldEncoder`.